# E3 consumer — GNN comparison analysis (no training in this notebook)

Loads the E3 producer pkls plus the E2 (QuIC probe + frozen splits) and
E7 (truncation curve) artifacts, and builds:

1. the headline table — QuIC linear-probe R2 vs trained-network test R2,
   per target, both n;
2. the protocol-parallel table — linear probe on each GNN's saved
   final-layer embeddings, same outer folds;
3. the collapse-diagnostic and tuning-selection tables;
4. the matched-dimension read-off — QuIC R2 at k bracketing WIDTH=64,
   straight from the E7 pkl, no new QuIC compute;
5. the cospectral mates through each GNN's embedding space.

Correction on record (producer header says otherwise): LapPE is NOT
provably blind to cospectral mates. Mates share eigenvalues; LapPE
features are eigenVECTORS, which differ between the graphs. No theorem
either way — item 5 measures it.

In [1]:
import pickle

import numpy as np
import networkx as nx

from collections import defaultdict
from itertools import combinations

from numpy.linalg import eigvalsh
from scipy.spatial.distance import pdist
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm

import warnings
from scipy.linalg import LinAlgWarning
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [2]:
SEED = 0
ALPHAS = np.logspace(-14, 2, 17)
NS = (14, 16)
MODELS = ('gin_const', 'gcn_const', 'gin_rni', 'gin_lappe')
WIDTH = 64

DATASET_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
E3_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14n16-e3-gnn-training-producer/"
E2_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14n16-e2-ridge-probe/"
E7_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14n16-e7-truncation/"
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}

In [3]:
def load_targets_and_spectra(n):
    with open(DATASET_BASES[n] + f"n{n}_data_dict.pkl", 'rb') as f:
        data_dict = pickle.load(f)
    keys = sorted(data_dict)
    assert len(keys) == EXPECTED_CUBIC_COUNTS[n]
    graphs = {k: nx.from_graph6_bytes(data_dict[k]['graph6'].encode())
              for k in keys}

    def field_or(field, fn, progress=False):
        if field in data_dict[keys[0]]:
            return np.array([data_dict[k][field] for k in keys], dtype=float)
        it = tqdm(keys, desc=f'n={n} {field}') if progress else keys
        return np.array([fn(k) for k in it], dtype=float)

    def c6_count(k):
        return sum(1 for c in nx.simple_cycles(graphs[k], length_bound=6)
                   if len(c) == 6)

    spectra = (np.stack([data_dict[k]['spectrum'] for k in keys])
               if 'spectrum' in data_dict[keys[0]]
               else np.stack([np.sort(eigvalsh(data_dict[k]['adj_mat']))
                              for k in keys]))
    targets = {
        'triangles':    field_or('num_triangles', None),
        '4-cycles':     field_or('num_4_cycles', None),
        '5-cycles':     field_or('num_5_cycles', None),
        '6-cycles':     field_or('num_6_cycles', c6_count, progress=True),
        'girth':        field_or('girth',    lambda k: nx.girth(graphs[k])),
        'diameter':     field_or('diameter', lambda k: nx.diameter(graphs[k])),
        'spectral_gap': spectra[:, -1] - spectra[:, -2],
    }
    return targets, spectra

TARGETS, SPECTRA = {}, {}
for n in NS:
    TARGETS[n], SPECTRA[n] = load_targets_and_spectra(n)

E3, E2, E7 = {}, {}, {}
for n in NS:
    E3[n] = {}
    for m in MODELS:
        with open(E3_BASE + f"n{n}_e3_{m}.pkl", 'rb') as f:
            E3[n][m] = pickle.load(f)
    with open(E2_BASE + f"n{n}_e2_ridge_probe.pkl", 'rb') as f:
        E2[n] = pickle.load(f)
    with open(E7_BASE + f"n{n}_e7_truncation.pkl", 'rb') as f:
        E7[n] = pickle.load(f)
    print(f'n={n}: loaded {len(E3[n])} model pkls + E2 + E7')

# splits come from the E2 pkl — the frozen record, not regenerated
SPLITS = {n: [(np.array(tr), np.array(te))
              for tr, te in E2[n]['splits']] for n in NS}
for n in NS:
    for m in MODELS:
        assert E3[n][m]['config']['splits'] == E2[n]['splits'], (
            f'n={n} {m}: producer splits differ from E2 — tables invalid')
    print(f'n={n}: producer splits match E2 splits')

n=14 num_6_cycles:   0%|          | 0/509 [00:00<?, ?it/s]

n=16 num_6_cycles:   0%|          | 0/4060 [00:00<?, ?it/s]

n=14: loaded 4 model pkls + E2 + E7
n=16: loaded 4 model pkls + E2 + E7
n=14: producer splits match E2 splits
n=16: producer splits match E2 splits


## 1. Headline table — trained-network test R2 vs the QuIC probe

QuIC column from the E2 pkl (linear probe on the full sorted vector);
GNN columns are each network's own test R2, trained on the target,
mean +- sd over 3 seeds x 5 folds. Negative values printed as-is.

In [4]:
HEADLINE = {}
for n in NS:
    HEADLINE[n] = {}
    print(f'\n=== n={n} ===')
    header = f'{"target":>13}   {"QuIC probe":>12}'
    for m in MODELS:
        header += f'   {m:>16}'
    print(header)
    for t in TARGETS[n]:
        quic = E2[n]['results'][t]
        row = {'quic': (quic['r2_mean'], quic['r2_sd'])}
        line = f'{t:>13}:  {quic["r2_mean"]:+.3f}+-{quic["r2_sd"]:.3f}'
        for m in MODELS:
            r2s = np.array([r['test_r2'] for r in E3[n][m]['runs'][t]])
            row[m] = (r2s.mean(), r2s.std())
            line += f'   {r2s.mean():+.3f}+-{r2s.std():.3f}'
        HEADLINE[n][t] = row
        print(line)


=== n=14 ===
       target     QuIC probe          gin_const          gcn_const            gin_rni          gin_lappe
    triangles:  +1.000+-0.000   -0.013+-0.026   -0.008+-0.009   +0.430+-0.052   +0.824+-0.240
     4-cycles:  +0.998+-0.002   -0.036+-0.050   -0.024+-0.028   +0.048+-0.076   +0.524+-0.258
     5-cycles:  +0.928+-0.092   -0.024+-0.024   -0.028+-0.027   -0.021+-0.027   +0.470+-0.132
     6-cycles:  +0.485+-0.326   -0.019+-0.024   -0.020+-0.029   +0.034+-0.035   +0.420+-0.094
        girth:  +0.993+-0.014   -0.015+-0.026   -0.016+-0.029   +0.140+-0.042   +0.820+-0.098
     diameter:  +0.548+-0.058   -0.023+-0.032   -0.023+-0.030   +0.268+-0.093   +0.664+-0.193
 spectral_gap:  +0.944+-0.010   -0.012+-0.023   -0.008+-0.009   +0.506+-0.068   +0.994+-0.004

=== n=16 ===
       target     QuIC probe          gin_const          gcn_const            gin_rni          gin_lappe
    triangles:  +1.000+-0.000   -0.003+-0.002   -0.002+-0.001   +0.566+-0.029   +0.941+-0.012
     4-cyc

## 2. Protocol parallelism — linear probe on GNN embeddings

Same protocol as the QuIC column: RidgeCV(cv=5, wide grid) on each
run's saved final-layer embeddings (width 64), fit on the run's outer
train fold, scored on its test fold, mean +- sd over runs. This is the
apples-to-apples column: fixed representation, linear readout, same
folds.

In [5]:
def probe_run(emb, y, tr, te):
    # small-alpha grid points probe below the embeddings'
    # numerical rank (const arms: rank 0 by construction);
    # inner CV rejects them — warning is advisory, suppressed
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', LinAlgWarning)
        model = RidgeCV(alphas=ALPHAS, cv=5)
    model.fit(emb[tr], y[tr])
    return r2_score(y[te], model.predict(emb[te]))

EMB_PROBE = {}
for n in NS:
    EMB_PROBE[n] = {}
    print(f'\n=== n={n} ===')
    header = f'{"target":>13}   {"QuIC probe":>12}'
    for m in MODELS:
        header += f'   {m:>16}'
    print(header)
    for t in TARGETS[n]:
        y = TARGETS[n][t]
        quic = E2[n]['results'][t]
        row = {}
        line = f'{t:>13}:  {quic["r2_mean"]:+.3f}+-{quic["r2_sd"]:.3f}'
        for m in MODELS:
            scores = []
            for r in E3[n][m]['runs'][t]:
                tr, te = SPLITS[n][r['fold']]
                scores.append(probe_run(r['embeddings'], y, tr, te))
            scores = np.array(scores)
            row[m] = (scores.mean(), scores.std())
            line += f'   {scores.mean():+.3f}+-{scores.std():.3f}'
        EMB_PROBE[n][t] = row
        print(line)


=== n=14 ===
       target     QuIC probe          gin_const          gcn_const            gin_rni          gin_lappe
    triangles:  +1.000+-0.000   -0.002+-0.004   -0.003+-0.004   +0.660+-0.054   +0.887+-0.106
     4-cycles:  +0.998+-0.002   -0.008+-0.006   -0.010+-0.014   +0.059+-0.042   +0.594+-0.103
     5-cycles:  +0.928+-0.092   -0.011+-0.012   -0.013+-0.014   -0.008+-0.021   +0.468+-0.161
     6-cycles:  +0.485+-0.326   -0.006+-0.008   -0.005+-0.008   +0.086+-0.052   +0.429+-0.099
        girth:  +0.993+-0.014   -0.012+-0.015   -0.008+-0.012   +0.210+-0.056   +0.783+-0.072
     diameter:  +0.548+-0.058   -0.008+-0.007   -0.008+-0.006   +0.387+-0.081   +0.720+-0.061
 spectral_gap:  +0.944+-0.010   -0.002+-0.002   -0.002+-0.002   +0.546+-0.057   +0.993+-0.003

=== n=16 ===
       target     QuIC probe          gin_const          gcn_const            gin_rni          gin_lappe
    triangles:  +1.000+-0.000   -0.001+-0.001   -0.001+-0.001   +0.718+-0.024   +0.948+-0.011
     4-cyc

## 3. Collapse diagnostics and tuning record

In [6]:
for n in NS:
    print(f'\n=== n={n} collapse ratio (max pair dist / mean norm) ===')
    print(f'{"target":>13}   ' + '   '.join(f'{m:>12}' for m in MODELS))
    for t in TARGETS[n]:
        line = f'{t:>13}: '
        for m in MODELS:
            ratios = np.array([r['collapse_ratio']
                               for r in E3[n][m]['runs'][t]])
            line += f'   {ratios.mean():>12.2e}'
        print(line)

    print(f'\n--- n={n} selected configs (depth, lr) + best epochs ---')
    for m in MODELS:
        sel = E3[n][m]['selected']
        eps = {t: int(np.median([r['best_epoch']
                                 for r in E3[n][m]['runs'][t]]))
               for t in sel}
        print(f'{m:>10}: ' + '  '.join(
            f'{t}=({sel[t]["depth"]},{sel[t]["lr"]},ep{eps[t]})'
            for t in sel))


=== n=14 collapse ratio (max pair dist / mean norm) ===
       target      gin_const      gcn_const        gin_rni      gin_lappe
    triangles:        0.00e+00       0.00e+00       7.00e-01       2.25e+00
     4-cycles:        0.00e+00       0.00e+00       7.42e-01       2.06e+00
     5-cycles:        0.00e+00       0.00e+00       3.27e-01       1.45e+00
     6-cycles:        0.00e+00       0.00e+00       8.20e-01       1.31e+00
        girth:        0.00e+00       0.00e+00       7.63e-01       2.27e+00
     diameter:        0.00e+00       0.00e+00       6.86e-01       1.32e+00
 spectral_gap:        0.00e+00       0.00e+00       1.53e+00       1.23e+00

--- n=14 selected configs (depth, lr) + best epochs ---
 gin_const: triangles=(3,0.001,ep8)  4-cycles=(3,0.01,ep12)  5-cycles=(3,0.001,ep8)  6-cycles=(5,0.01,ep21)  girth=(3,0.01,ep20)  diameter=(3,0.001,ep8)  spectral_gap=(3,0.01,ep26)
 gcn_const: triangles=(3,0.01,ep12)  4-cycles=(3,0.01,ep7)  5-cycles=(3,0.001,ep5)  6-cycles=(5,0.0

## 4. Matched dimension — QuIC at k ~ WIDTH, from the E7 curve

No new QuIC compute: the E7 grid brackets width 64 at k=50 and k=100.

In [7]:
for n in NS:
    ks = [k for k in E7[n]['ks'] if k in (50, 100)]
    print(f'\n=== n={n}: QuIC truncated-probe R2 at k bracketing '
          f'WIDTH={WIDTH} ===')
    print(f'{"target":>13}   ' + '   '.join(f'k={k:>4}' for k in ks))
    for t in ['triangles', '4-cycles', '5-cycles', '6-cycles']:
        vals = [E7[n]['results'][k][t]['r2_mean'] for k in ks]
        print(f'{t:>13}:  ' + '   '.join(f'{v:+.3f}' for v in vals))


=== n=14: QuIC truncated-probe R2 at k bracketing WIDTH=64 ===
       target   k=  50   k= 100
    triangles:  +1.000   +1.000
     4-cycles:  +0.988   +0.993
     5-cycles:  -0.026   +0.272
     6-cycles:  +0.228   +0.235

=== n=16: QuIC truncated-probe R2 at k bracketing WIDTH=64 ===
       target   k=  50   k= 100
    triangles:  +1.000   +1.000
     4-cycles:  +0.981   +0.996
     5-cycles:  +0.167   +0.335
     6-cycles:  +0.179   +0.184


## 5. Cospectral mates through each GNN's embedding space

Mates discovered by spectrum hashing (seconds). Per model: mate-pair
embedding distance over median all-pair embedding distance, averaged
over the triangles-target runs. Const arms are trivially 0/0 —
reported for completeness. The LapPE number is the empirical answer to
the corrected question above: eigenvector features are not shared
between mates, so nothing forbids separation; do the trained models
actually separate them?

In [8]:
MATES = {}
for n in NS:
    spec_hash = [tuple(np.round(s, 8)) for s in SPECTRA[n]]
    groups = defaultdict(list)
    for k, h in enumerate(spec_hash):
        groups[h].append(k)
    MATES[n] = [p for g in groups.values() if len(g) > 1
                for p in combinations(g, 2)]
    print(f'n={n}: {len(MATES[n])} cospectral pair(s): {MATES[n]}')

MATE_EMB = {}
for n in NS:
    if not MATES[n]:
        continue
    MATE_EMB[n] = {}
    print(f'\n=== n={n}: mate embedding separation / median all-pair, '
          f'triangles-target runs ===')
    print(f'{"pair":>14}   ' + '   '.join(f'{m:>12}' for m in MODELS))
    per_pair = {p: {} for p in MATES[n]}
    for m in MODELS:
        for i, j in MATES[n]:
            vals = []
            for r in E3[n][m]['runs']['triangles']:
                emb = r['embeddings'].astype(np.float64)
                med = np.median(pdist(emb))
                d = np.linalg.norm(emb[i] - emb[j])
                vals.append(d / (med + 1e-300))
            per_pair[(i, j)][m] = float(np.mean(vals))
    for p in MATES[n]:
        print(f'{str(p):>14}   ' + '   '.join(
            f'{per_pair[p][m]:>12.3e}' for m in MODELS))
    MATE_EMB[n] = per_pair

n=14: 3 cospectral pair(s): [(79, 458), (201, 203), (234, 239)]
n=16: 43 cospectral pair(s): [(1, 5), (24, 35), (82, 885), (83, 884), (106, 632), (259, 268), (282, 1494), (320, 3059), (457, 516), (476, 3741), (479, 967), (494, 568), (582, 3061), (582, 3800), (3061, 3800), (583, 3067), (586, 1502), (594, 601), (710, 711), (817, 1130), (936, 940), (1034, 1116), (1035, 1123), (1075, 1086), (1181, 3517), (1371, 1792), (1496, 1570), (1505, 1506), (1710, 1734), (1898, 3951), (2121, 3077), (2209, 2226), (2276, 3578), (2428, 2431), (2693, 3900), (3047, 3520), (3058, 3799), (3090, 3525), (3162, 3248), (3514, 3515), (3602, 4005), (3672, 3689), (3693, 3902)]

=== n=14: mate embedding separation / median all-pair, triangles-target runs ===
          pair      gin_const      gcn_const        gin_rni      gin_lappe
     (79, 458)      0.000e+00      0.000e+00      9.542e-01      1.806e-01
    (201, 203)      0.000e+00      0.000e+00      9.359e-01      2.336e-01
    (234, 239)      0.000e+00      0.

In [9]:
for n in NS:
    with open(f'/kaggle/working/n{n}_e3_analysis.pkl', 'wb') as f:
        pickle.dump({'headline': HEADLINE[n],
                     'embedding_probe': EMB_PROBE[n],
                     'mate_embedding': MATE_EMB.get(n, {}),
                     'mates': MATES[n],
                     'quic_source': 'n{}_e2_ridge_probe.pkl'.format(n),
                     'e7_source': 'n{}_e7_truncation.pkl'.format(n)}, f)
    print(f'saved n{n}_e3_analysis.pkl')

saved n14_e3_analysis.pkl
saved n16_e3_analysis.pkl


## Results

(Written after the run: the fang paragraph — const rows at <= 0 with
collapse ratio 0 against the QuIC column; whether RNI/LapPE lift off
zero on any cycle target and at what cost; the diameter row — the
honest GNN-wins cell if it lands; the matched-dimension sentence; and
what the mates look like through LapPE.)

## E3 results — summary and reading

**The comparison, in one sentence.** An untrained quantum embedding read
with a linear probe beats task-trained, tuned, seed-averaged GNNs given
every advantage on every cycle-count target at both scales, while the
theoretically-bounded GNN class is not merely worse but exactly blind.

**Const arms: provably constant, and measured so.** Across all 210
trained const-arm models (2 architectures x 7 targets x 3 seeds x 5
folds), the collapse ratio — max pairwise embedding distance over mean
embedding norm, computed exactly in float64 — is 0.00e+00. Not small:
zero. On regular graphs with uniform node features, every 1-WL round
assigns every vertex the identical color, so GIN and GCN compute a
constant function of the graph; the measurement confirms the argument
to the bit. Test R2 sits at the constant-predictor floor (slightly
negative under CV, approaching 0 from below as test folds grow at
n=16), and median best-epochs of 5–26 record that early stopping found
nothing to train. The verb "provably incapable" belongs to these two
columns and ONLY these two columns.

**RNI: symmetry broken, ladder barely climbed.** With features
resampled every epoch, noise-averaged validation, an extended budget
(best epochs 250–414 at n=16 — it used the budget), and eval averaged
over 10 draws, RNI reads triangles partially (net 0.57, probe 0.72 at
n=16) and almost nothing deeper: C4 0.19–0.20, C5 0.10–0.27, C6 ~0.07–
0.12. Universality-in-expectation is an expressiveness statement, not a
learnability one, and at 3–4k training graphs the gap between the two
is the whole story. Verb: "fails to match despite every advantage."

**LapPE: the strongest competitor, and it obeys our ordering.** LapPE
nearly closes triangles (0.94 net at n=16), then degrades monotonically
down the cycle ladder — 0.61 (C4), 0.57 (C5), 0.50 (C6) — against QuIC's
1.000 / 1.000 / 0.982 / 0.642. The best fix available to the competitor
class exhibits the same accessibility-depth ordering the paper
characterizes in the embedding. Caveat on record: LapPE's C4 variance
is large (+-0.26 across seeds/folds, plausibly eigenvector sign-flip
instability); reported as-is.

**The two cells the GNNs win, which make the table credible.** Diameter
(LapPE 0.807 vs QuIC 0.737 at n=16) and spectral gap (0.996 vs 0.932).
Both non-cycle targets, both outside the theory's protected claims, one
of them (diameter) pre-registered in the design discussion as the
plausible honest loss. The advantage arms win exactly where nothing
says they shouldn't and lose everywhere the characterization says they
must — the opposite signature of a rigged benchmark.

**Matched dimension, honest in both directions.** From the E7 curve,
QuIC truncated to k=50 sorted amplitudes — untrained, 50 numbers —
outscores every width-64 trained arm on triangles (1.000) and C4
(0.98+). But C5 and C6 at k~64 are weak (0.17–0.34): QuIC's deep-layer
advantage requires readout depth, because deeper cycle content lives
further down the sorted vector. Both halves get stated; either alone
invites the cherry-pick reading.

**Scale replication (E4, embedded in the QuIC column).** Every n=14
decodability number holds or improves at n=16: C5 0.928+-0.092 ->
0.982+-0.015 (the weak fold was thin-sample noise, resolved as
predicted), C6 0.485 -> 0.642, diameter 0.548 -> 0.737, C4's residual
gone (1.000). C6 still frays (+-0.122): layer four remains conditional
even at 4060 graphs, which the E1 stratification result predicts rather
than apologizes for.

**Cospectral mates through the competitors' eyes.** Const arms: blind
by collapse. RNI: mates at ~0.9 of the median all-pair embedding
distance — indistinguishable from ordinary pairs; symmetry breaking
carries no structural preference. LapPE: mates land closer than typical
but clearly separated (0.18–0.52 of median). Correction on record: the
producer header's claim that LapPE is provably blind to mates is wrong —
mates share eigenVALUES while LapPE consumes eigenVECTORS, which differ
between the graphs; there is no theorem either way, so this is an
empirical answer to an empirical question.

**The n=16 census.** 43 cospectral pairs, including one mutually
cospectral TRIPLE (582, 3061, 3800). The count-identical reference
class and the tracer analysis gain real statistical mass at n=16, and
the triple is an exhibit in its own right.

**Artifacts.** n{14,16}_e3_analysis.pkl (this notebook);
n{n}_e3_{model}.pkl (producer, 8 files); QuIC column from
n{n}_e2_ridge_probe.pkl; matched-dimension from n{n}_e7_truncation.pkl.
All tables on the frozen E2 splits, asserted identical before anything
printed.

The comparison's structure matters as much as its margins. Where the QuIC embedding is outscored — diameter and spectral gap — it trails by 0.06–0.07, and it trails task-trained networks consuming Laplacian eigenvector features: spectral inputs, spectral targets, specialist per target. Where the message-passing class is outscored, the failure is categorical: with uniform features its representation is provably constant on this input family (measured collapse: 0.00e+00), and its strongest repair recovers the shallowest cycle layer while fading monotonically with depth. Across every target at both scales, the fixed, untrained embedding's weakest linear readout (R² = 0.64) exceeds what three of the four trained competitor arms achieve on half the table. The distinction we characterize is not that the embedding scores higher on average, but that it has no empty cells: every target examined is decodable to a substantial degree from a single representation computed once, with no training signal.
